In [1]:
!pip install cvxpy

     |████████████████████████████████| 1.2 MB 3.6 MB/s eta 0:00:01
     |████████████████████████████████| 300 kB 29.7 MB/s eta 0:00:01
     |████████████████████████████████| 221 kB 31.5 MB/s eta 0:00:01
     |████████████████████████████████| 10.4 MB 24.1 MB/s eta 0:00:01
     |████████████████████████████████| 1.3 MB 80.8 MB/s eta 0:00:01
     |████████████████████████████████| 1.1 MB 78.6 MB/s eta 0:00:01
You should consider upgrading via the '/share/home2/ebrahimis/project1_3_8/venv/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import numpy as np
import cvxpy as cp

# Define problem parameters (example values)
U = 10  # Number of users
R = 5   # Number of RUs
K = 20  # Number of PRBs
T = 100 # Number of time steps
D = 2   # Number of DUs
C = 1   # Number of CUs

# Example values for constants
P_max_rd = np.ones(R) * 10  # Max power for each RU
W_E2_dc_RIC = np.ones(D) * 5000  # Bandwidth for each DU to RIC
W_FH_rd_dc = np.ones(R) * 50000    # Bandwidth for each RU to DU
B_s = np.ones(U) * 10  # Required bit rate
D_s = np.ones(U) * 5   # Maximum tolerable delay

# Sigmoid parameters
xi = 2
omega = 0.5
vartheta = np.ones(U)  # Weight of each slice type


In [9]:
# Variables (flattened)
chi = cp.Variable(R * U * T, boolean=True)
rho = cp.Variable(R * K * U * T, boolean=True)
p = cp.Variable(R * K * U * T)

# Helper Functions for Indexing
def index_chi(r, u, t):
    return r * U * T + u * T + t

def index_rho(r, k, u, t):
    return r * K * U * T + k * U * T + u * T + t

def index_p(r, k, u, t):
    return r * K * U * T + k * U * T + u * T + t

# Sigmoid-like functions
def sigmoid_bitrate(bitrate, B_s, xi):
    return (bitrate / B_s)**xi / (1 + (bitrate / B_s)**xi)

def sigmoid_delay(delay, D_s, xi):
    return (D_s / delay)**xi / (1 + (D_s / delay)**xi)

# Aggregated SSL function for each user
def SSL_u(rate_ssl, delay_ssl, omega):
    return (rate_ssl**omega) * (delay_ssl**(1 - omega))

# Overall SSL function
def overall_SSL(SSL_u, vartheta):
    weighted_SSL = SSL_u**vartheta.reshape(-1, 1)  # Reshape vartheta for broadcasting
    return cp.prod(weighted_SSL, axis=0) ** (1 / sum(vartheta))

In [16]:
import cvxpy as cp
import numpy as np

# Constants
R = 3  # Number of RATs
K = 2  # Number of PRBs
U = 4  # Number of Users
T = 5  # Number of Time Slots
C = 2  # Number of Cores
D = 2  # Number of DCs

# Variables (flattened)
chi = cp.Variable(R * U * T, boolean=True)
rho = cp.Variable(R * K * U * T, boolean=True)
p = cp.Variable(R * K * U * T)

# Helper Functions for Indexing
def index_chi(r, u, t):
    return r * U * T + u * T + t

def index_rho(r, k, u, t):
    return r * K * U * T + k * U * T + u * T + t

def index_p(r, k, u, t):
    return r * K * U * T + k * U * T + u * T + t

# Sigmoid-like functions
def sigmoid_bitrate(bitrate, B_s, xi):
    return (bitrate / B_s)**xi / (1 + (bitrate / B_s)**xi)

def sigmoid_delay(delay, D_s, xi):
    return (D_s / delay)**xi / (1 + (D_s / delay)**xi)

# Aggregated SSL function for each user
def SSL_u(rate_ssl, delay_ssl, omega):
    return (rate_ssl**omega) * (delay_ssl**(1 - omega))

# Overall SSL function
def overall_SSL(SSL_u, vartheta):
    weighted_SSL = SSL_u**vartheta[:, None]  # Reshape vartheta for broadcasting
    product_SSL = np.prod(weighted_SSL, axis=0)  # Use np.prod instead of cp.prod
    sum_vartheta = np.sum(vartheta)  # Use np.sum instead of cp.sum
    return product_SSL**(1 / sum_vartheta)

# Constraints list
constraints = []

# C1 and C2 constraints (assuming zeta values are predefined)
zeta_cd = np.ones((C, D))  # Placeholder values
zeta_dc_rd = np.ones((D, R))  # Placeholder values

for d in range(D):
    constraints += [cp.sum(zeta_cd[:, d]) == 1]

for r in range(R):
    constraints += [cp.sum(zeta_dc_rd[:, r]) == 1]

# C3 constraint
for u in range(U):
    for t in range(T):
        constraints += [cp.sum([chi[index_chi(r, u, t)] for r in range(R)]) == 1]

# # C4 constraint
# for u in range(U):
#     for t in range(T):
#         constraints += [cp.sum([chi[index_chi(r, u, t)] for r in range(R)]) == 1]

# C5 constraint
for r in range(R):
    for k in range(K):
        for t in range(T):
            constraints += [cp.sum([chi[index_chi(r, u, t)] * rho[index_rho(r, k, u, t)] for u in range(U)]) <= 1]

# C6 constraint
P_max_rd = np.ones(R)  # Placeholder
for r in range(R):
    for t in range(T):
        constraints += [cp.sum([chi[index_chi(r, u, t)] * rho[index_rho(r, k, u, t)] * p[index_p(r, k, u, t)] 
                                for k in range(K) for u in range(U)]) <= P_max_rd[r]]

# C7 constraint
W_E2_dc_RIC = np.ones(D)  # Placeholder
b = np.random.rand(R, U, T)  # Placeholder
for d in range(D):
    for t in range(T):
        constraints += [cp.sum([zeta_cd[c, d] * zeta_dc_rd[d, r] * chi[index_chi(r, u, t)] * b[r, u, t] 
                                for c in range(C) for r in range(R) for u in range(U)]) <= W_E2_dc_RIC[d]]

# C8 constraint
W_FH_rd_dc = np.ones(R)  # Placeholder
for r in range(R):
    for t in range(T):
        constraints += [cp.sum([zeta_dc_rd[d, r] * chi[index_chi(r, u, t)] * b[r, u, t] 
                                for d in range(D) for u in range(U)]) <= W_FH_rd_dc[r]]

# C9 constraint
D_s = np.ones((U, T))  # Placeholder
B_s = 1  # Placeholder
xi = 1  # Placeholder
omega = 0.5  # Placeholder
vartheta = np.ones(U)  # Placeholder
rate_ssl = sigmoid_bitrate(b, B_s, xi)
delay_ssl = sigmoid_delay(D_s, D_s, xi)  # Placeholder
ssl_u = SSL_u(rate_ssl, delay_ssl, omega)
ssl = overall_SSL(ssl_u, vartheta)
constraints += [ssl >= 0.5]

# C10 and C11 constraints are already handled by boolean variable definition
# C12 constraint
P_max = 1  # Placeholder
for r in range(R):
    for k in range(K):
        for u in range(U):
            for t in range(T):
                constraints += [p[index_p(r, k, u, t)] >= 0, p[index_p(r, k, u, t)] <= P_max]

# SCA iterations
max_iterations = 100
tolerance = 1e-3
ssl_prev = 0

for it in range(max_iterations):
    # Define SSL over all time steps
    ssl_overall = cp.sum(ssl)

    # Define objective
    objective = cp.Maximize(ssl_overall)
    
    # Solve the problem
    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.MOSEK)  # Use an appropriate solver
    
    # Check for convergence
    ssl_curr = problem.value
    if abs(ssl_curr - ssl_prev) < tolerance:
        break
    ssl_prev = ssl_curr

    # Update the linearizations/approximations if necessary
    # (This part depends on how non-convexity is handled; you may need specific updates here)

# Extract the optimized variables
chi_opt = np.array([chi.value[index_chi(r, u, t)] for r in range(R) for u in range(U) for t in range(T)]).reshape(R, U, T)
rho_opt = np.array([rho.value[index_rho(r, k, u, t)] for r in range(R) for k in range(K) for u in range(U) for t in range(T)]).reshape(R, K, U, T)
p_opt = np.array([p.value[index_p(r, k, u, t)] for r in range(R) for k in range(K) for u in range(U) for t in range(T)]).reshape(R, K, U, T)

print("Optimal User Assignment (chi):", chi_opt)
print("Optimal PRB Allocation (rho):", rho_opt)
print("Optimal Transmission Power (p):", p_opt)


ValueError: Problem has an invalid constraint of type <class 'numpy.ndarray'>